PubMed Journal Article Fetcher for Google Colab
This notebook fetches articles from specified journals for a given month using PubMed API

In [1]:
# Install required packages

!pip install biopython pandas requests

import os, pandas as pd, requests, time, warnings, json, re
from Bio import Entrez
from datetime import datetime, timedelta
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional, Tuple
import calendar
# warnings.filterwarnings('ignore')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/3.2 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.3 MB/s  0:00:00


# Instructions for use

📋 INSTRUCTIONS:

1. Update the EMAIL variable with your email address (required by NCBI)
2. Adjust JOURNALS and LOOKBACK_DAYS to shape the search window
3. Run the cells to execute main() and harvest only new PMIDs within the rolling window or the requested month

⚠️  IMPORTANT NOTES:

- Use your real email address - it’s required by NCBI’s usage policy
- Journal names should match PubMed’s format exactly
- Large queries may take several minutes to complete
- Be respectful of API rate limits
- Previously harvested PMIDs are stored in data/ent_search/seen_pmids.json to avoid duplicates
- When running via GitHub Actions, you can supply TARGET_MONTH (YYYY-MM) or YEAR + MONTH inputs to rerun a specific calendar month instead of the rolling LOOKBACK_DAYS window.

🚀 To start, run: main()


In [2]:
# Parameters
EMAIL = os.getenv("UNPAYWALL_EMAIL", "")

JOURNALS = [
    "International Forum of Allergy & Rhinology",
    "Rhinology",
    "JAMA Otolaryngology–Head & Neck Surgery",
    "Otolaryngology–Head and Neck Surgery",
    "European Annals of Otorhinolaryngology–Head and Neck Diseases",
    "Journal of Voice",
    "American Journal of Rhinology & Allergy",
    "JARO – Journal of the Association for Research in Otolaryngology",
    "Journal of Otolaryngology–Head & Neck Surgery",
    "Laryngoscope",
    "Auris Nasus Larynx",
    "new england journal of medicine",
    "JAMA"
]

LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", "30"))
TARGET_MONTH = os.getenv("TARGET_MONTH", "").strip()
TARGET_YEAR = os.getenv("TARGET_YEAR", "").strip()
TARGET_MONTH_NUMBER = os.getenv("TARGET_MONTH_NUMBER", "").strip()
OUTPUT_DIR = os.path.join("data", "ent_search")

ALLOW_EMPTY_HARVEST = os.getenv("ALLOW_EMPTY_HARVEST", "false").lower() == "true"


In [3]:
# Parameters
TARGET_MONTH = "2026-03"
TARGET_YEAR = ""
TARGET_MONTH_NUMBER = ""


In [4]:
from datetime import datetime

RUN_STARTED_AT = datetime.now()


In [5]:
#!/usr/bin/env python3

MONTH_TOKEN_PATTERN = re.compile(r"\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\b", re.IGNORECASE)


def load_seen_pmids(path: str) -> set:
    """Load a set of previously seen PMIDs from disk."""
    if os.path.exists(path):
        try:
            with open(path) as f:
                data = json.load(f)
                if isinstance(data, list):
                    return set(map(str, data))
        except Exception as exc:
            print(f"⚠️  Could not load seen PMIDs: {exc}")
    return set()


def save_seen_pmids(path: str, pmids: set) -> None:
    """Persist a set of previously seen PMIDs to disk."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(sorted(pmids), f, indent=2)


def compute_requested_window(target_month: str, target_year: str, target_month_number: str):
    """Return a (start_date, end_date) tuple for a requested calendar month."""
    target_month = (target_month or "").strip()
    target_year = (target_year or "").strip()
    target_month_number = (target_month_number or "").strip()

    if target_month:
        try:
            start_date = datetime.strptime(target_month, "%Y-%m")
        except ValueError as exc:
            raise ValueError(f"TARGET_MONTH must be YYYY-MM; received '{target_month}'") from exc
        end_day = calendar.monthrange(start_date.year, start_date.month)[1]
        end_date = start_date.replace(day=end_day, hour=23, minute=59, second=59)
        return start_date, end_date

    if target_year and target_month_number:
        try:
            year_int = int(target_year)
            month_int = int(target_month_number)
            start_date = datetime(year_int, month_int, 1)
        except ValueError as exc:
            raise ValueError(
                f"TARGET_YEAR and TARGET_MONTH_NUMBER must form a valid date; received '{target_year}-{target_month_number}'"
            ) from exc
        end_day = calendar.monthrange(year_int, month_int)[1]
        end_date = start_date.replace(day=end_day, hour=23, minute=59, second=59)
        return start_date, end_date

    return None


def build_publication_date(pub_date: dict) -> str:
    year = str(pub_date.get("Year", "")).strip()
    month = str(pub_date.get("Month", "")).strip()
    day = str(pub_date.get("Day", "")).strip()
    medline_date = str(pub_date.get("MedlineDate", "")).strip()

    if not year and medline_date:
        year_match = re.search(r"(19|20)\d{2}", medline_date)
        month_match = MONTH_TOKEN_PATTERN.search(medline_date)
        day_match = re.search(r"\b([0-3]?\d)\b", medline_date)
        if year_match:
            year = year_match.group(0)
        if month_match:
            month = month_match.group(1).title()
        if day_match:
            day = day_match.group(1)

    publication_date = "Unknown"
    if year:
        publication_date = year
        if month:
            publication_date += f"-{month}"
        if day:
            publication_date += f"-{day}"

    return publication_date


def normalize_publication_date(date_str: str) -> str:
    date_str = str(date_str or "").strip()
    if not date_str or date_str == "Unknown":
        return ""

    for fmt in ("%Y-%b-%d", "%Y-%b", "%Y-%m-%d", "%Y-%m"):
        try:
            return datetime.strptime(date_str, fmt).date().isoformat()
        except ValueError:
            continue

    try:
        return datetime.fromisoformat(date_str).date().isoformat()
    except ValueError:
        return ""


def extract_publication_month(date_str: str) -> str:
    normalized = normalize_publication_date(date_str)
    return normalized[:7] if normalized else ""


def write_failed_pmids_report(
    path: str,
    target_month: str,
    failed_pmids: List[Dict[str, str]],
    filtered_out_pmids: List[Dict[str, str]],
) -> None:
    if not failed_pmids and not filtered_out_pmids:
        if os.path.exists(path):
            os.remove(path)
        return

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(
            {
                "target_month": target_month,
                "failed_pmids": failed_pmids,
                "filtered_out_pmids": filtered_out_pmids,
            },
            f,
            indent=2,
        )


class PubMedFetcher:

    def __init__(self, email: str):
        """Initialize with email for API requests."""
        self.email = email
        Entrez.email = email
        self.request_delay = 0.34  # ~3 requests per second max
        self.max_attempts = 3

    def search_articles(
        self,
        journals: List[str],
        lookback_days: int,
        start_date_override: Optional[datetime] = None,
        end_date_override: Optional[datetime] = None,
    ):
        """Search for articles in specified journals within a rolling or fixed window."""
        end_date = end_date_override or datetime.now()
        start_date = start_date_override or end_date - timedelta(days=lookback_days)

        start_date_str = start_date.strftime("%Y/%m/%d")
        end_date_str = end_date.strftime("%Y/%m/%d")

        journal_query = " OR ".join([f'\"{journal}\"[Journal]' for journal in journals]) if journals else ""

        query_parts = []
        if journal_query:
            query_parts.append(f"({journal_query})")
        query_parts.append(f"({start_date_str}[PDAT] : {end_date_str}[PDAT])")

        final_query = " AND ".join(query_parts)

        try:
            print(f"Querying PubMed with: {final_query}")
            handle = Entrez.esearch(db="pubmed", term=final_query, datetype="pdat", retmax=100000)
            record = Entrez.read(handle)
            handle.close()
            time.sleep(self.request_delay)

            id_list = list(dict.fromkeys(record["IdList"]))
            print(f"Found {len(id_list)} articles")
            return id_list, start_date_str, end_date_str

        except Exception as e:
            print(f"Error searching PubMed: {e}")
            return [], start_date_str, end_date_str

    def fetch_article_details(self, pmids: List[str]) -> Tuple[List[Dict], List[Dict[str, str]]]:
        """Fetch detailed information for a list of PubMed IDs."""
        articles: List[Dict] = []
        failures: List[Dict[str, str]] = []

        for pmid in pmids:
            last_error = ""
            for attempt in range(1, self.max_attempts + 1):
                handle = None
                try:
                    handle = Entrez.efetch(db="pubmed", id=pmid, retmode="xml")
                    time.sleep(self.request_delay)
                    record = Entrez.read(handle)

                    if not record or "PubmedArticle" not in record:
                        last_error = "No article data returned"
                        print(f"⚠️  No article data found for PMID: {pmid}")
                        break

                    article = record["PubmedArticle"][0]
                    journal_info = article["MedlineCitation"]["Article"]["Journal"]
                    journal = str(journal_info.get("Title", "Unknown Journal"))
                    title = str(article["MedlineCitation"]["Article"].get("ArticleTitle", "No Title"))

                    abstract = ""
                    if "Abstract" in article["MedlineCitation"]["Article"]:
                        abstract_texts = article["MedlineCitation"]["Article"].get("AbstractText", [])
                        abstract = " ".join(str(part) for part in abstract_texts)

                    authors = []
                    if "AuthorList" in article["MedlineCitation"]["Article"]:
                        for author in article["MedlineCitation"]["Article"]["AuthorList"]:
                            lastname = author.get("LastName", "")
                            firstname = author.get("ForeName", "")
                            fullname = f"{firstname} {lastname}".strip()
                            if fullname:
                                authors.append(fullname)

                    pub_date = journal_info.get("JournalIssue", {}).get("PubDate", {})
                    publication_date = build_publication_date(pub_date)

                    doi = ""
                    for article_id in article["PubmedData"].get("ArticleIdList", []):
                        if article_id.attributes.get("IdType") == "doi":
                            doi = str(article_id)
                            break

                    volume = journal_info.get("JournalIssue", {}).get("Volume", "")
                    issue = journal_info.get("JournalIssue", {}).get("Issue", "")
                    pages = article["MedlineCitation"]["Article"].get("Pagination", {}).get("MedlinePgn", "")

                    articles.append({
                        "PMID": pmid,
                        "Title": title,
                        "Journal": journal,
                        "Authors": "; ".join(authors),
                        "PublicationDate": publication_date,
                        "Volume": volume,
                        "Issue": issue,
                        "Pages": pages,
                        "DOI": doi,
                        "Abstract": abstract[:500] + "..." if len(abstract) > 500 else abstract,
                    })
                    last_error = ""
                    break
                except Exception as e:
                    last_error = str(e)
                    if attempt == self.max_attempts:
                        print(f"⚠️  Failed to fetch PMID {pmid} after {self.max_attempts} attempts: {e}")
                    else:
                        backoff = min(5.0, self.request_delay * (2 ** attempt))
                        print(
                            f"⚠️  Attempt {attempt}/{self.max_attempts} failed for PMID {pmid}: {e}. Retrying in {backoff:.2f}s..."
                        )
                        time.sleep(backoff)
                finally:
                    if handle is not None:
                        try:
                            handle.close()
                        except Exception:
                            pass

            if last_error:
                failures.append({"pmid": pmid, "error": last_error})

        return articles, failures


def main():

    print("=== PubMed Journal Article Fetcher ===")

    print(f"Email: {EMAIL}")
    if JOURNALS:
        print(f"Journals: {', '.join(JOURNALS)}")
    else:
        print("No journal filter configured; searching across all journals.")
    print("-" * 50)

    requested_window = compute_requested_window(TARGET_MONTH, TARGET_YEAR, TARGET_MONTH_NUMBER)
    if requested_window:
        start_dt, end_dt = requested_window
        requested_month_key = start_dt.strftime("%Y-%m")
        print(
            f"Requested month window: {start_dt.strftime('%Y-%m-%d')} to {end_dt.strftime('%Y-%m-%d')} (from TARGET_MONTH/TARGET_YEAR+TARGET_MONTH_NUMBER)"
        )
        print(f"Target publication month: {requested_month_key}")
    else:
        end_dt = datetime.now()
        start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)
        requested_month_key = ""
        print(f"Rolling window: last {LOOKBACK_DAYS} days ({start_dt.strftime('%Y-%m-%d')} to {end_dt.strftime('%Y-%m-%d')})")
    print("-" * 50)

    if EMAIL == "your.email@example.com":
        print("⚠️  Please update the EMAIL variable with your actual email address!")
        print("This is required by NCBI's API usage policy.")
        return

    fetcher = PubMedFetcher(EMAIL)

    print("🔍 Searching for articles...")
    pmid_list, start_date, end_date = fetcher.search_articles(
        JOURNALS,
        LOOKBACK_DAYS,
        start_dt,
        end_dt,
    )

    if not pmid_list:
        print("❌ No articles found matching the criteria.")
        print(f"Summary: discovered {len(pmid_list)} PMIDs between {start_date} and {end_date}.")
        if ALLOW_EMPTY_HARVEST:
            print("⚠️  Empty harvest allowed via ALLOW_EMPTY_HARVEST flag; exiting without failure.")
            return
        raise RuntimeError("No articles found for the configured search window.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    seen_pmids_path = os.path.join(OUTPUT_DIR, "seen_pmids.json")
    failed_report_path = os.path.join(OUTPUT_DIR, "ent_failed_pmids.json")
    seen_pmids = load_seen_pmids(seen_pmids_path)
    if seen_pmids:
        print(f"Loaded {len(seen_pmids)} previously harvested PMIDs.")

    if requested_month_key:
        new_pmids = list(dict.fromkeys(pmid_list))
        print(f"PMIDs to fetch for targeted month rerun: {len(new_pmids)}")
    else:
        new_pmids = [pmid for pmid in pmid_list if pmid not in seen_pmids]
        print(f"PMIDs to fetch after filtering seen set: {len(new_pmids)}")

    if not new_pmids:
        print("⚠️  No new PMIDs to process within this window.")
        print(f"Summary: discovered {len(pmid_list)} PMIDs, seen {len(seen_pmids)}, new {len(new_pmids)} between {start_date} and {end_date}.")
        if ALLOW_EMPTY_HARVEST:
            print("⚠️  Empty harvest allowed via ALLOW_EMPTY_HARVEST flag; exiting without failure.")
            return
        raise RuntimeError("No new PMIDs to process after filtering seen set.")

    print(f"📖 Fetching details for {len(new_pmids)} candidate articles...")
    articles, failed_pmids = fetcher.fetch_article_details(new_pmids)

    if not articles:
        print("❌ Failed to fetch article details for all candidate PMIDs.")
        write_failed_pmids_report(failed_report_path, requested_month_key, failed_pmids, [])
        if ALLOW_EMPTY_HARVEST:
            print("⚠️  Empty harvest allowed via ALLOW_EMPTY_HARVEST flag; exiting without failure.")
            return
        raise RuntimeError("No article details were successfully fetched.")

    filtered_out_pmids: List[Dict[str, str]] = []
    if requested_month_key:
        filtered_articles = []
        for article in articles:
            publication_month = extract_publication_month(article.get("PublicationDate", ""))
            if publication_month == requested_month_key:
                filtered_articles.append(article)
            else:
                filtered_out_pmids.append(
                    {
                        "pmid": str(article.get("PMID", "")).strip(),
                        "publication_date": str(article.get("PublicationDate", "")).strip(),
                        "publication_month": publication_month,
                    }
                )
        print(
            f"Kept {len(filtered_articles)} articles in publication month {requested_month_key}; filtered out {len(filtered_out_pmids)} cross-month records."
        )
        articles = filtered_articles

    write_failed_pmids_report(
        failed_report_path,
        requested_month_key,
        failed_pmids,
        filtered_out_pmids,
    )

    if not articles:
        print("❌ No articles remained after publication-month filtering.")
        if ALLOW_EMPTY_HARVEST:
            print("⚠️  Empty harvest allowed via ALLOW_EMPTY_HARVEST flag; exiting without failure.")
            return
        raise RuntimeError("No articles remained after filtering to the requested publication month.")

    df = pd.DataFrame(articles)

    print(f"✅ Successfully retrieved {len(articles)} articles!")
    print(f"Columns: {', '.join(df.columns.tolist())}")
    print("First 5 articles:")
    print(df.head().to_string(max_colwidth=50))

    window_label = f"{start_date.replace('/', '-')}_to_{end_date.replace('/', '-')}"
    csv_path = os.path.join(OUTPUT_DIR, f"ent_raw_results_{window_label}.csv")
    df.to_csv(csv_path, index=False)

    successful_pmids = {str(article.get("PMID", "")).strip() for article in articles if str(article.get("PMID", "")).strip()}
    if successful_pmids:
        updated_seen_pmids = seen_pmids | successful_pmids
        save_seen_pmids(seen_pmids_path, updated_seen_pmids)
        print(f"Updated seen PMIDs saved to: {seen_pmids_path}")

    json_path = os.path.join(OUTPUT_DIR, "ent_all_results.json")
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)

    print("📊 Summary:")
    print(f"Total new articles: {len(articles)}")
    print(f"Unique journals: {df['Journal'].nunique()}")
    if failed_pmids:
        print(f"Failed PMID fetches after retries: {len(failed_pmids)}")
    if filtered_out_pmids:
        print(f"Cross-month records excluded: {len(filtered_out_pmids)}")
    print("Articles per journal:")
    journal_counts = df['Journal'].value_counts()
    for journal, count in journal_counts.head(10).items():
        print(f"  • {journal}: {count}")

    print(f"💾 Raw CSV saved to: {csv_path}")
    print(f"💾 ENT results JSON saved to: {json_path}")
    if failed_pmids or filtered_out_pmids:
        print(f"💾 Partial-harvest report saved to: {failed_report_path}")

    return df


In [6]:
# Optional dry-run preview for the last 30 days
# Set DRY_RUN_PREVIEW=true in the environment to exercise this without editing code.
DRY_RUN_PREVIEW = os.getenv("DRY_RUN_PREVIEW", "false").lower() == "true"

if DRY_RUN_PREVIEW:
    if not EMAIL:
        raise ValueError("EMAIL must be configured to run the dry-run search.")
    dry_run_end = datetime.now()
    dry_run_start = dry_run_end - timedelta(days=30)
    print(f"Dry-run window: {dry_run_start.strftime('%Y-%m-%d')} to {dry_run_end.strftime('%Y-%m-%d')}")
    print("Running dry-run search...")
    fetcher = PubMedFetcher(EMAIL)
    pmid_list, start_date, end_date = fetcher.search_articles(
        JOURNALS,
        LOOKBACK_DAYS,
        dry_run_start,
        dry_run_end,
    )
    print(f"Dry-run returned {len(pmid_list)} PMIDs.")
    print("Sample PMIDs:", pmid_list[:10])
else:
    print("Dry run disabled. Set DRY_RUN_PREVIEW=true to log a quick query.")


Dry run disabled. Set DRY_RUN_PREVIEW=true to log a quick query.


In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
df = main()
if df is not None:
    print("=== DataFrame Summary ===")
    print(df.describe(include='all'))


=== PubMed Journal Article Fetcher ===
Email: shvecht@gmail.com
Journals: International Forum of Allergy & Rhinology, Rhinology, JAMA Otolaryngology–Head & Neck Surgery, Otolaryngology–Head and Neck Surgery, European Annals of Otorhinolaryngology–Head and Neck Diseases, Journal of Voice, American Journal of Rhinology & Allergy, JARO – Journal of the Association for Research in Otolaryngology, Journal of Otolaryngology–Head & Neck Surgery, Laryngoscope, Auris Nasus Larynx, new england journal of medicine, JAMA
--------------------------------------------------
Requested month window: 2026-03-01 to 2026-03-31 (from TARGET_MONTH/TARGET_YEAR+TARGET_MONTH_NUMBER)
Target publication month: 2026-03
--------------------------------------------------
🔍 Searching for articles...
Querying PubMed with: ("International Forum of Allergy & Rhinology"[Journal] OR "Rhinology"[Journal] OR "JAMA Otolaryngology–Head & Neck Surgery"[Journal] OR "Otolaryngology–Head and Neck Surgery"[Journal] OR "European A

Found 545 articles
PMIDs to fetch for targeted month rerun: 545
📖 Fetching details for 545 candidate articles...


⚠️  Attempt 1/3 failed for PMID 41670977: Remote end closed connection without response. Retrying in 0.68s...


Kept 536 articles in publication month 2026-03; filtered out 9 cross-month records.
✅ Successfully retrieved 536 articles!
Columns: PMID, Title, Journal, Authors, PublicationDate, Volume, Issue, Pages, DOI, Abstract
First 5 articles:
       PMID                                              Title               Journal                                            Authors PublicationDate Volume Issue    Pages                        DOI Abstract
0  41921449  Successful management of near-infrared photoim...  Auris, nasus, larynx  Mayu Shigeyama; Naoki Nishio; Akihisa Wada; Se...     2026-Mar-31     53     3  375-379  10.1016/j.anl.2026.03.009         
1  41917688  Value-Based Neuromonitoring in Thyroidectomy: ...      The Laryngoscope  Daqi Zhang; Francesco Brucchi; Carla Colombo; ...     2026-Mar-31                               10.1002/lary.70536         
2  41916771  Correction to "Washing Illness Away: A Systema...      The Laryngoscope                                                    